# Projection-target prediction from expression

Can a cell's **injection site** (Frontal Cortex / Cerebellum / Spinal Cord) be predicted from its expression profile? We compare a random per-cell split (optimistic) against a mouse-grouped split (batch-honest) to see how much of the signal is a per-animal batch effect.

`re_run_all=True` re-runs the full grid search; set it to `False` to load previously saved results.

In [ ]:
re_run_all = True

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
import os
notebook_dir = os.path.dirname(os.path.abspath(''))
if notebook_dir not in sys.path:
    sys.path.append(os.path.dirname(notebook_dir))
from notebooks.config import *
configure_matplotlib()
SAVE_FIGURES = True
fig_path = RETROSEQ_FIGURE_DIR
from functools import partia
l
save_figure = partial(save_figure, dir_path=fig_path)
cpm_scl = CPM_SCL

In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import glob, os, sys
import scanpy as sc
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier

import processing, utils, plotting, heatmap
import pseudoclusters, test_structures
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
import pickle
print(f"scanpy version: {sc.__version__}")


## Load data

Load the filtered retroseq object and keep the three projection targets (drop thalamus / unlabeled cells).

In [ ]:
savefilename = RETROSEQ_DATA_DIR+"/retroseq_updated_filtered.h5ad"
adata_retro = sc.read_h5ad(savefilename)
adata_retro = adata_retro[adata_retro.obs['injection_site'].notna(), :]

# keep the three projection targets (drop thalamus)
adata_retro = adata_retro[adata_retro.obs['injection_site'] != 'thalamus', :]
adata_retro.obs['injection_site'] = adata_retro.obs['injection_site'].cat.remove_unused_categories()
adata_retro.shape

In [ ]:

mapping = {
    cat: cat.title()
    for cat in adata_retro.obs['injection_site'].cat.categories}
    
adata_retro.obs['injection_site'] = (
    adata_retro.obs['injection_site']
    .cat.rename_categories(mapping))

## Features, labels, and train/val/test split

Standardize the expression matrix (`X`), label-encode the injection site (`y`), record each cell's mouse of origin (for grouped CV later), then make a stratified 60 / 20 / 20 train / val / test split.

In [ ]:
# X = adata_retro.layers['BN']
X = np.array(adata_retro.X.toarray())
scaler = StandardScaler().fit(X)
X = scaler.transform(X)

le = LabelEncoder()
y = le.fit_transform(adata_retro.obs['injection_site'])

# mouse identity per cell ("batch"); each mouse maps to exactly one injection site
groups = adata_retro.obs['external_donor_name'].astype(str).values

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.40,
    stratify=y,
    random_state=15
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,     # half of 40% → 20% each
    stratify=y_temp,
    random_state=15
)


In [ ]:
# Prepare X/Y lists for convenience
X_list = [X_train, X_val, X_test]
y_list = [y_train, y_val, y_test]


## Per-cell classification (main result)

In [ ]:
if re_run_all:
    
    # Define a helper to train, validate, and test
    def fit_and_report(name, X_list, y_list, model, param_grid=None, scale=False):
        X_tr, X_val_, X_te = X_list
        y_tr, y_val_, y_te = y_list
        steps = []
        if scale:
            steps.append(('scaler', StandardScaler()))
        steps.append(('clf', model))
        pipe = Pipeline(steps)
        # Hyperparameter tuning if grid given
        if param_grid is not None:
            cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=15)
            grid = GridSearchCV(pipe, param_grid, cv=cv, n_jobs=-1, scoring='accuracy')
            grid.fit(X_tr, y_tr)
            best = grid.best_estimator_
            print(f"{name} best params: {grid.best_params_}")
        else:
            best = pipe.fit(X_tr, y_tr)
        # Validation
        y_val_pred = best.predict(X_val_)
        val_acc = accuracy_score(y_val_, y_val_pred)
        print(f"[{name}] Val Accuracy: {val_acc:.4f}")
        # Retrain on train+val then test
        best.fit(np.vstack([X_tr, X_val_]), np.concatenate([y_tr, y_val_]))
        y_test_pred = best.predict(X_te)
        test_acc = accuracy_score(y_te, y_test_pred)
        print(f"[{name}] Test Accuracy: {test_acc:.4f}\n")
        return name, best, val_acc, test_acc, y_test_pred
    
    
    
    # random_state pinned on the stochastic estimators (RandomForest, SVC, GradBoost)
    # so re-runs are reproducible; matches the mouse-grouped CV cell below.
    models = {
        'LogisticRegression': LogisticRegression(max_iter=1000),
        'RandomForest':        RandomForestClassifier(random_state=15),
        'SVC':                 SVC(random_state=15),
        'KNN':                 KNeighborsClassifier(),
        'GradBoost':           GradientBoostingClassifier(random_state=15),  # note gradient boosting takes > 1 hours
        # 'XGBoost':             XGBClassifier(use_label_encoder=False, eval_metric='logloss')
    }
    param_grids = {
        'LogisticRegression': {'clf__C': [0.01, 0.1, 1, 10]},
        'RandomForest':       {'clf__n_estimators': [100, 200], 'clf__max_depth': [None, 10, 20]},
        'SVC':                {'clf__C': [0.1, 1, 10], 'clf__kernel': ['linear', 'rbf']},
        'KNN':                {'clf__n_neighbors': [3, 5, 7]},
        'GradBoost':          {
            'clf__n_estimators': [100, 200],
            'clf__learning_rate':[0.01, 0.1],
            'clf__max_depth':    [3, 5]
        },

    }
    
    results = []
    for name, model in models.items():
        print(f"Running {name}...")
        pg = param_grids.get(name)
        m_name, best, val_acc, test_acc, y_pred = fit_and_report(
            name, X_list, y_list, model, pg, scale=True
        )
        results.append({
            'model':    m_name,
            'val_acc':  val_acc,
            'test_acc': test_acc,
            'best':     best,
            'y_pred':   y_pred
        })
    
    results_df = pd.DataFrame(results)[['model','val_acc','test_acc']]
    display(results_df)
    
    
    
    plt.figure(figsize=(6,4))
    plt.bar(results_df['model'], results_df['test_acc'])
    plt.ylim(0,1)
    plt.ylabel('Test Accuracy')
    plt.title('Model Comparison')
    plt.show()
    
    
    # classification report for the best model 
    best_idx = np.argmax([r['test_acc'] for r in results])
    best_row = results[best_idx]
    print(f"Best model: {best_row['model']}")
    print(classification_report(y_test, best_row['y_pred'], target_names=le.classes_))

    import pickle 
    output_file = os.path.join(OUTPUT_DIR, 'retroseq_prediction_results.pkl')
    with open(output_file, "wb") as f:
        pickle.dump(results, f)

    
else:
    print('skipping the grid search for each model since this takes forever, just loading the results saved!')

### Confusion matrix (per-cell, GradBoost)

In [ ]:
######### loading #########

input_file = os.path.join(OUTPUT_DIR, 'retroseq_prediction_results.pkl')
# input_file = '/root/capsule/output/retroseq_prediction_results.pkl'
with open(input_file, "rb") as f:
    results = pickle.load(f)




######### plotting #########
model_name = 'GradBoost'   # ← change as desired
row = next(r for r in results if r['model'] == model_name)
y_pred = row['y_pred']
cm_full = confusion_matrix(y_test, y_pred)

# 3) Define your desired class order
new_order = ['Frontal Cortex', 'Cerebellum', 'Spinal Cord']
# new_order = ['frontal cortex', 'cerebellum', 'spinal cord']
# Map those names back to the integer labels
order_idx = [list(le.classes_).index(c) for c in new_order]


# Row-normalize over ALL predicted classes first, THEN subset to the display order, so any
# dropped/extra class can never be renormalized away into a spurious confusion.
cm_pct_full = cm_full / cm_full.sum(axis=1, keepdims=True)
cm_pct = cm_pct_full[np.ix_(order_idx, order_idx)]
overall_acc = accuracy_score(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6,6))
im = ax.imshow(cm_pct, cmap='Blues', vmin=0, vmax=1)
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, shrink = .3)
# cbar.set_label('Fraction of true class', rotation=270, labelpad=15)

# Tick labels in your new order
ax.set_xticks(np.arange(len(new_order)))
ax.set_yticks(np.arange(len(new_order)))
ax.set_xticklabels(new_order, rotation=45, ha='right')
ax.set_yticklabels(new_order)
ax.set_ylabel('True label')
ax.set_xlabel('Predicted label')

# Title with overall accuracy
ax.set_title(f'Overall accuracy: {overall_acc:.2%}')

# Annotate each cell with its percentage
for i in range(len(new_order)):
    for j in range(len(new_order)):
        fontcol = 'black' if cm_pct[i,j]<.7 else 'white'
        ax.text(
            j, i,
            f"{cm_pct[i, j]*100:.1f}%",
            ha='center', va='center', color=fontcol)

plt.tight_layout()
src_cm = pd.DataFrame(cm_pct, index=pd.Index(new_order, name="true_label"),
                      columns=pd.Index(new_order, name="predicted_label"))
save_figure("fig_s11a", source_data=src_cm)
# plt.gcf().set_dpi(300)

print("Analysis complete! Confusion matrix have been saved to:", fig_path)

## Generalization across mice

Grouped by mouse (no animal shared between train and test), accuracy falls toward chance — projection target is not predictable across animals with the current cohort (3–8 mice per target).

In [ ]:

print('mice per site:')
print(adata_retro.obs.groupby('injection_site')['external_donor_name'].nunique())

# 3 = max splits given 3 mice in the smallest classes; no mouse split across train/test
sgkf = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=15)
for tr, te in sgkf.split(X, y, groups):
    assert set(groups[tr]).isdisjoint(set(groups[te])), 'a mouse leaked across train/test!'
print('OK: no mouse is split across train/test in any fold\n')

# fixed (reasonable) hyperparameters so the grouped CV is fast and self-contained
models_grp = {
    'LogisticRegression': LogisticRegression(max_iter=1000, C=1),
    'RandomForest':       RandomForestClassifier(n_estimators=200, random_state=15),
    'SVC':                SVC(C=1, kernel='rbf'),
    'KNN':                KNeighborsClassifier(n_neighbors=5),
    'GradBoost':          GradientBoostingClassifier(random_state=15),
}
grouped = {}
for name, model in models_grp.items():
    pipe = Pipeline([('scaler', StandardScaler()), ('clf', model)])
    yhat = cross_val_predict(pipe, X, y, cv=sgkf, groups=groups, n_jobs=-1)
    grouped[name] = {'acc': accuracy_score(y, yhat), 'y_pred': yhat}
    print(f'[{name}] mouse-grouped CV accuracy: {grouped[name]["acc"]:.4f}')

In [ ]:
# random per-cell split (leaky) vs mouse-grouped CV (batch-honest)
rand_acc = {r['model']: r['test_acc'] for r in results}   # `results` loaded/produced above
cmp_df = pd.DataFrame([{'model': m,
                        'random_split_acc': rand_acc.get(m, np.nan),
                        'mouse_grouped_acc': grouped[m]['acc']} for m in grouped])
display(cmp_df)

xpos = np.arange(len(cmp_df)); w = 0.38
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(xpos - w/2, cmp_df['random_split_acc'], w, label='random per-cell split (leaky)', color='#bdbdbd')
ax.bar(xpos + w/2, cmp_df['mouse_grouped_acc'], w, label='mouse-grouped CV (batch-honest)', color='#2c7fb8')
ax.axhline(1/len(np.unique(y)), ls='--', c='k', lw=1, label='chance')
ax.set_xticks(xpos); ax.set_xticklabels(cmp_df['model'], rotation=45, ha='right')
ax.set_ylim(0, 1); ax.set_ylabel('accuracy'); ax.legend(fontsize=7)
ax.set_title('Projection-target prediction: the accuracy drop is the batch effect')
plt.tight_layout(); save_figure('fig_S11a', source_data=cmp_df.set_index('model')); plt.show()

out = os.path.join(OUTPUT_DIR, 'retroseq_prediction_grouped_results.pkl')
with open(out, 'wb') as f:
    pickle.dump({'cv_strategy': 'StratifiedGroupKFold(n_splits=3, group=external_donor_name)',
                 'grouped': grouped, 'comparison': cmp_df}, f)
print('saved', out)